In [1]:
import sys
#sys.path.append('C:/Users/michael.croquette/Documents/github/pyrpl') # replace with your pyrpl path
import pyrpl

In [2]:
HOSTNAME = "192.168.68.172"


p = pyrpl.Pyrpl(hostname=HOSTNAME, config="",reloadfpga=True,gui=True)
APP.start()

INFO:pyrpl.redpitaya:Found FPGA binfile at: c:\Users\BERRY4\Desktop\Russell_New\Code\pyrpl-main\pyrpl\pyrpl\fpga/red_pitaya.bin
INFO:pyrpl.redpitaya:Found DTBO file at: c:\Users\BERRY4\Desktop\Russell_New\Code\pyrpl-main\pyrpl\pyrpl\fpga/red_pitaya.dtbo
INFO:pyrpl.redpitaya:Installing the OS 3 compatible hard-float monitor_server at //opt//pyrpl//monitor_server
INFO:pyrpl.redpitaya:Successfully connected to Redpitaya with hostname 192.168.68.172.


NameError: name 'APP' is not defined

In [ ]:
# Route the signal generated by IQ0 to the Red Pitaya's analog output 1

iq0 = p.rp.iq0
square = p.rp.square
square.input = 'iq0'
#iq0.output_direct = "out1"

In [ ]:
# Check that the scope input selector registers accept the Python DSP map.
# This briefly changes each selector register, then restores its exact prior value.
scope = p.rp.scope

for channel in ("input1", "input2"):
    register = getattr(type(scope), channel)
    address = register.address
    options = register.options(scope)
    original = scope._read(address)

    print(f"{channel}: initial FPGA value={original}")
    try:
        for signal in ("in1", "in2", "square"):
            expected = options[signal]
            scope._write(address, expected)
            actual = scope._read(address)
            status = "PASS" if actual == expected else "FAIL"
            print(f"  {signal}: expected={expected}, read back={actual} [{status}]")
            assert actual == expected, (
                f"{channel} register did not retain {signal!r}: "
                f"expected {expected}, got {actual}"
            )
    finally:
        scope._write(address, original)
        restored = scope._read(address)
        print(f"  restored FPGA value={restored}")
        assert restored == original, (
            f"Could not restore {channel}: expected {original}, got {restored}"
        )